# Phase 4: Monte Carlo Playoff Simulation

Brief overview: projecting 2026-27 team strength from blended 
historical performance and current-roster playoff experience, building 
playoff seeding, and simulating the postseason bracket thousands of 
times to estimate championship odds per team.

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import joblib

## 2. Load Phase 2/3 Outputs

In [2]:
# loading team season feats from phase 2, and the trained model from phase 3
games_regular_season_grouped = pd.read_csv(r'D:\Data Science Projects\nba-26-27-predictor\data\processed\team_season_features.csv')
playoff_experience_snapshot = pd.read_csv(r'D:\Data Science Projects\nba-26-27-predictor\data\processed\playoff_experience_snapshot.csv')
log_reg = joblib.load(r'D:\Data Science Projects\nba-26-27-predictor\models\logistic_regression_model.pkl')

print(games_regular_season_grouped.shape)
print(playoff_experience_snapshot.shape)

(60, 15)
(30, 2)


## 3. Blend Historical Seasons into a Projected Strength Score

**Question:** How do we combine 2024-25 and 2025-26 team performance 
into a single projected strength score for 2026-27, weighting recent 
performance more heavily?

We weight 2025-26 at 70% and 2024-25 at 30% — recent form is more 
predictive of near-future performance than older data, consistent with 
our project's original recency-over-sample-size design choice. This 
reflects franchise-level performance continuity; it does not account 
for individual player movement beyond the playoff experience feature 
(a scoping decision appropriate to a 2-season, team-level dataset, not 
an oversight).

In [3]:
# pivoting so each team has one row w/ separate columns for each seasons feats
strength_cols = ['WIN_PERCENTAGE', 'eFG%', 'TOV%', 'ORB%', 'FT Rate', 'NET_RATING', 
                  'mvp_share', 'ALL_NBA_COUNT', 'ALL_NBA_SHARE_SUM']

season_2425 = games_regular_season_grouped[games_regular_season_grouped['SEASON_LABEL'] == '2024-25'].set_index('TEAM_ABBREVIATION')[strength_cols]
season_2526 = games_regular_season_grouped[games_regular_season_grouped['SEASON_LABEL'] == '2025-26'].set_index('TEAM_ABBREVIATION')[strength_cols]

# weighted blend: 70/30 
RECENT_WEIGHT = 0.70
PRIOR_WEIGHT = 0.30

projected_strength = (season_2526 * RECENT_WEIGHT) + (season_2425 * PRIOR_WEIGHT)
projected_strength = projected_strength.reset_index()

display(projected_strength.head())

,TEAM_ABBREVIATION,WIN_PERCENTAGE,eFG%,TOV%,ORB%,FT Rate,NET_RATING,mvp_share,ALL_NBA_COUNT,ALL_NBA_SHARE_SUM
0,ATL,0.539024,0.552359,0.119197,0.247925,0.187537,1.362773,0.0000,0.7,0.1750
1,BKN,0.265854,0.519659,0.133566,0.237201,0.208118,-9.193637,0.0000,0.0,0.0000
2,BOS,0.701220,0.556042,0.104160,0.280003,0.170747,8.413683,0.1556,1.0,0.8376
3,CHA,0.445122,0.538638,0.129093,0.293192,0.193813,0.725640,0.0000,0.0,0.0000
4,CHI,0.407317,0.549562,0.125400,0.227674,0.187286,-3.949391,0.0000,0.0,0.0000


## 4. Layer in Roster-Aware Playoff Experience

**Question:** How do we incorporate the one feature that already 
reflects current 2026-27 rosters, rather than relying purely on 
franchise-level historical continuity?

In [4]:
# merging in the current-roster playoff experience snapshot
projected_strength = pd.merge(
    projected_strength, 
    playoff_experience_snapshot, 
    on='TEAM_ABBREVIATION',
    how='left'
)

display(projected_strength.head())

,TEAM_ABBREVIATION,WIN_PERCENTAGE,eFG%,TOV%,ORB%,FT Rate,NET_RATING,mvp_share,ALL_NBA_COUNT,ALL_NBA_SHARE_SUM,TOTAL_PLAYOFF_EXPERIENCE
0,ATL,0.539024,0.552359,0.119197,0.247925,0.187537,1.362773,0.0000,0.7,0.1750,189.0
1,BKN,0.265854,0.519659,0.133566,0.237201,0.208118,-9.193637,0.0000,0.0,0.0000,60.0
2,BOS,0.701220,0.556042,0.104160,0.280003,0.170747,8.413683,0.1556,1.0,0.8376,198.0
3,CHA,0.445122,0.538638,0.129093,0.293192,0.193813,0.725640,0.0000,0.0,0.0000,46.0
4,CHI,0.407317,0.549562,0.125400,0.227674,0.187286,-3.949391,0.0000,0.0,0.0000,22.0


## 5. Compute Win Probability Against a League-Average Opponent

**Question:** How do we turn each team's projected features into a 
single, comparable strength score using the model we already trained?

Rather than inventing a new arbitrary weighting formula, we reuse our 
trained Logistic Regression model — feeding each team's projected 
features against a "league-average opponent" row gives us a model-based 
win probability, which doubles as a ranking metric for seeding.

In [5]:
# building a league-average opponent row serving as a constant baseline
league_avg = projected_strength[strength_cols + ['TOTAL_PLAYOFF_EXPERIENCE']].mean()

# building the model input, each teams own feats + the league avg
model_input_rows = []

for _, row in projected_strength.iterrows():
    model_row = {
        'WIN_PERCENTAGE': row['WIN_PERCENTAGE'],
        'eFG%': row['eFG%'],
        'TOV%': row['TOV%'],
        'ORB%': row['ORB%'],
        'FT Rate': row['FT Rate'],
        'NET_RATING': row['NET_RATING'],
        'mvp_share': row['mvp_share'],
        'ALL_NBA_COUNT': row['ALL_NBA_COUNT'],
        'ALL_NBA_SHARE_SUM': row['ALL_NBA_SHARE_SUM'],
        'OPP_WIN_PERCENTAGE': league_avg['WIN_PERCENTAGE'],
        'OPP_eFG%': league_avg['eFG%'],
        'OPP_TOV%': league_avg['TOV%'],
        'OPP_ORB%': league_avg['ORB%'],
        'OPP_FT Rate': league_avg['FT Rate'],
        'OPP_NET_RATING': league_avg['NET_RATING'],
        'OPP_mvp_share': league_avg['mvp_share'],
        'OPP_ALL_NBA_COUNT': league_avg['ALL_NBA_COUNT'],
        'OPP_ALL_NBA_SHARE_SUM': league_avg['ALL_NBA_SHARE_SUM'],
        'OPP_PLAYOFF_ROUND_REACHED': 0,
        'OPP_WON_CHAMPIONSHIP': False
    }
    model_input_rows.append(model_row)

model_input_df = pd.DataFrame(model_input_rows)

# predicting win prob against a league-average opponent for every team
projected_strength['POWER_SCORE'] = log_reg.predict_proba(model_input_df)[:, 1]

display(projected_strength[['TEAM_ABBREVIATION', 'POWER_SCORE']].sort_values('POWER_SCORE', ascending=False))

,TEAM_ABBREVIATION,POWER_SCORE
20,OKC,0.810748
2,BOS,0.727324
8,DET,0.688831
5,CLE,0.674492
19,NYK,0.663286
7,DEN,0.659277
26,SAS,0.650426
10,HOU,0.639744
17,MIN,0.598461
13,LAL,0.579569


## 6. Incorporate Roster Value into Power Score

**Question:** How do we combine our model-based POWER_SCORE (from 
historical franchise performance) with TOTAL_ROSTER_VALUE (from current, 
roster-aware individual player performance) into one final ranking that 
reflects real 2026-27 team strength?

In [6]:
# loading roster value snapshot 
roster_value_snapshot = pd.read_csv(r'D:\Data Science Projects\nba-26-27-predictor\data\processed\roster_value_snapshot.csv')


# merging onto projected_strength
projected_strength = pd.merge(
    projected_strength,
    roster_value_snapshot[['TEAM_ABBREVIATION', 'TOTAL_ROSTER_VALUE']],
    on='TEAM_ABBREVIATION',
    how='left'
)

display(projected_strength[['TEAM_ABBREVIATION', 'POWER_SCORE', 'TOTAL_ROSTER_VALUE']].head())

,TEAM_ABBREVIATION,POWER_SCORE,TOTAL_ROSTER_VALUE
0,ATL,0.536996,288.307717
1,BKN,0.240556,253.200387
2,BOS,0.727324,274.496397
3,CHA,0.481668,279.116478
4,CHI,0.374026,260.443007


In [7]:
# percentile rank instead of min-max — robust to outlier teams stretching/compressing the scale
projected_strength['ROSTER_VALUE_NORMALIZED'] = projected_strength['TOTAL_ROSTER_VALUE'].rank(pct=True)

# final blended power score: 40% model, 60% roster value
FINAL_POWER_WEIGHT_HISTORICAL = 0.40
FINAL_POWER_WEIGHT_ROSTER = 0.60

projected_strength['FINAL_POWER_SCORE'] = (
    projected_strength['POWER_SCORE'] * FINAL_POWER_WEIGHT_HISTORICAL +
    projected_strength['ROSTER_VALUE_NORMALIZED'] * FINAL_POWER_WEIGHT_ROSTER
)

display(projected_strength[['TEAM_ABBREVIATION', 'FINAL_POWER_SCORE']].sort_values('FINAL_POWER_SCORE', ascending=False))

,TEAM_ABBREVIATION,FINAL_POWER_SCORE
26,SAS,0.800170
22,PHI,0.777527
24,POR,0.764475
7,DEN,0.763711
21,ORL,0.762231
10,HOU,0.715898
19,NYK,0.685315
13,LAL,0.671828
5,CLE,0.649797
20,OKC,0.644299
